## Example Ring and Sidechain Datasets

In [2]:
import pandas as pd
import pyarrow.parquet as pq
import pyarrow as pa
from pathlib import Path

# Define the directory where the files will be saved
HOME_DIR = Path.cwd()  # Change this if needed
DATA_DIR = HOME_DIR / "data"
DATA_DIR.mkdir(exist_ok=True)  # Ensure the directory exists

# Create Rings Dataset
rings_data = [
    {"id": "R1", "smiles": "A", "n_subs": 2},  # Benzene with 2 attachment points
    {"id": "R2", "smiles": "B", "n_subs": 3},  # Pyridine with 3 attachment points
    {"id": "R3", "smiles": "C", "n_subs": 1},  # Pyridine with 1 attachment point
    {"id": "R3", "smiles": "D", "n_subs": 5},  # Pyridine with 1 attachment point
    {"id": "R3", "smiles": "E", "n_subs": 4},  # Pyridine with 1 attachment point
    {"id": "R3", "smiles": "F", "n_subs": 4},  # Pyridine with 1 attachment point
    {"id": "R3", "smiles": "G", "n_subs": 4},  # Pyridine with 1 attachment point
    {"id": "R3", "smiles": "H", "n_subs": 4},  # Pyridine with 1 attachment point
    {"id": "R3", "smiles": "I", "n_subs": 4},  # Pyridine with 1 attachment point
    {"id": "R3", "smiles": "J", "n_subs": 4},  # Pyridine with 1 attachment point
]

# Convert to DataFrame
df_rings = pd.DataFrame(rings_data)

# Save as Parquet
df_rings.to_parquet(DATA_DIR / "ring_list.parquet", engine="pyarrow")

# Create Sidechains Dataset
sidechains_data = [
    {"id": "S1", "smiles": "WE"},   # Methoxy (-OCH3)
    {"id": "S2", "smiles": "ARE"},  # Chloromethyl (-CH2Cl)
    {"id": "S2", "smiles": "SO"},  # Chloromethyl (-CH2Cl)
    {"id": "S2", "smiles": "BACK"},  # Chloromethyl (-CH2Cl)
    {"id": "S2", "smiles": "NOT"},  # Chloromethyl (-CH2Cl)
    {"id": "S2", "smiles": "YET"},  # Chloromethyl (-CH2Cl)
    {"id": "S2", "smiles": "TILL"},  # Chloromethyl (-CH2Cl)
    {"id": "S2", "smiles": "THAT"},  # Chloromethyl (-CH2Cl)
]

# Convert to DataFrame
df_sidechains = pd.DataFrame(sidechains_data)

# Save as Parquet
df_sidechains.to_parquet(DATA_DIR / "sidechain_list.parquet", engine="pyarrow")

print(f"✅ Rings and Sidechains datasets saved in {DATA_DIR}")


✅ Rings and Sidechains datasets saved in C:\Users\zachg\PycharmProjects\BillionMolecules\billion_mol_db\testing_recombination\data


## Recombination Logic

Original logic that parses parquet files incorrectly

In [3]:
import duckdb
import pandas as pd
import concurrent.futures
import pyarrow as pa
from pathlib import Path
import pyarrow.parquet as pq
import itertools
import os
import shutil


def new_combination(ring_smiles, sub_smiles_list):
    """Generates a new combined SMILES string from a ring and multiple sidechains."""
    return ring_smiles + "(" + ")(".join(sub_smiles_list) + ")"


# Function to process and write each batch to its own file
def process_and_write_batch(df_batch, parquet_schema, batch_id, temp_dir):
    df_batch["combination"] = df_batch.apply(
        lambda row: new_combination(row["ring_smiles"], [row[f"sub{i}_smiles"] for i in range(1, row["n_subs"] + 1)]),
        axis=1,
    )

    df_batch = df_batch.drop(columns=["n_subs"])  # Remove n_subs column after processing

    # Convert DataFrame to Apache Arrow table
    table = pa.Table.from_pandas(df_batch, schema=parquet_schema)

    # Write to a temporary Parquet file
    temp_file = os.path.join(temp_dir, f"batch_{batch_id}.parquet")
    pq.write_table(table, temp_file)


def main():
    # Variable and file paths
    HOME_DIR = Path.cwd()  # Gets the current working directory
    # HOME_DIR = Path(__file__).resolve().parent.parent
    ring_file = HOME_DIR / "data" / "ring_list.parquet"
    sidechain_file = HOME_DIR / "data" / "sidechain_list.parquet"
    mol_db_file = HOME_DIR / "data" / "mol_db.parquet"
    temp_dir = HOME_DIR / "temp_batches"  # Directory to store temporary batch files
    temp_dir.mkdir(exist_ok=True)

    id_name = "id"
    smiles_name = "smiles"
    n_subs_name = "n_subs"
    batch_size = 1000  # Adjust this batch size as necessary

    # Load the molecule fragments from a file into DuckDB
    con = duckdb.connect()
    con.execute(f"CREATE TABLE rings AS SELECT * FROM '{ring_file}'")  # Load rings into DuckDB
    con.execute(f"CREATE TABLE sidechains AS SELECT * FROM '{sidechain_file}'")  # Load sidechains into DuckDB

    # Set up Parquet schema dynamically based on the maximum number of substitutions
    max_subs = con.execute(f"SELECT MAX({n_subs_name}) FROM rings").fetchone()[0]
    parquet_schema_fields = [
        ("ring_id", pa.string()),
        ("ring_smiles", pa.string()),
        ("n_subs", pa.int32()),  # Keep track of how many substitutions
    ]
    for i in range(1, max_subs + 1):
        parquet_schema_fields.append((f"sub{i}_id", pa.string()))
        parquet_schema_fields.append((f"sub{i}_smiles", pa.string()))
    parquet_schema_fields.append(("combination", pa.string()))  # Final combined SMILES
    parquet_schema = pa.schema(parquet_schema_fields)

    # Create the base query for rings and the first sidechain (must be an INNER JOIN to guarantee at least one sidechain)
    query = f"""
        SELECT rings.{id_name} AS ring_id, rings.{smiles_name} AS ring_smiles, rings.{n_subs_name} AS n_subs,
               side1.{id_name} AS sub1_id, side1.{smiles_name} AS sub1_smiles
        FROM rings
        INNER JOIN sidechains AS side1 ON TRUE
        WHERE rings.{n_subs_name} >= 1
        """
    
    # Dynamically add additional LEFT JOINs for more sidechains based on `n_subs`
    for i in range(2, max_subs + 1):
        query += f"""
        LEFT JOIN sidechains AS side{i} ON rings.{n_subs_name} >= {i}
        """
    
    print("Generated SQL Query:\n", query)  # Debugging: Check query before execution

    # Execute the query and use fetch_df_chunk to fetch data in chunks
    cursor = con.execute(query)

    executor = concurrent.futures.ProcessPoolExecutor()
    batch_id = 0

    while True:
        # Fetch the next chunk of data using fetch_df_chunk()
        df_batch = cursor.fetch_df_chunk(batch_size)
        if df_batch.empty:
            break  # No more data

        # Process each batch in parallel and write to its own file
        future = executor.submit(
            process_and_write_batch, df_batch=df_batch, parquet_schema=parquet_schema, batch_id=batch_id, temp_dir=temp_dir
        )

        batch_id += 1

    # After processing, merge all temp files into the final file
    with pq.ParquetWriter(mol_db_file, parquet_schema) as writer:
        for temp_file in temp_dir.glob("*.parquet"):
            table = pq.read_table(temp_file)
            writer.write_table(table)

    total_rows = con.execute(f"SELECT COUNT(*) FROM ({query})").fetchone()[0]
    print(f"Mol DB with {total_rows} rows written to {mol_db_file}.")

    # Cleanup temporary files and connections
    shutil.rmtree(temp_dir)  # Remove entire temp directory
    con.close()
    executor.shutdown()


if __name__ == "__main__":
    main()


Generated SQL Query:
 
        SELECT rings.id AS ring_id, rings.smiles AS ring_smiles, rings.n_subs AS n_subs,
               side1.id AS sub1_id, side1.smiles AS sub1_smiles
        FROM rings
        INNER JOIN sidechains AS side1 ON TRUE
        WHERE rings.n_subs >= 1
        
        LEFT JOIN sidechains AS side2 ON rings.n_subs >= 2
        
        LEFT JOIN sidechains AS side3 ON rings.n_subs >= 3
        
        LEFT JOIN sidechains AS side4 ON rings.n_subs >= 4
        
        LEFT JOIN sidechains AS side5 ON rings.n_subs >= 5
        


ParserException: Parser Error: syntax error at or near "LEFT"

New start from scratch logic ensuring we can use duckdb to load the data from parquet files correctly

In [18]:
import duckdb
import pandas as pd
from pathlib import Path

# Define file paths
HOME_DIR = Path.cwd()
ring_file = HOME_DIR / "data" / "ring_list.parquet"
sidechain_file = HOME_DIR / "data" / "sidechain_list.parquet"
combinations_file = HOME_DIR / "data" / "combinations_2.parquet"

# Connect to DuckDB
con = duckdb.connect()

# Load Parquet files into DuckDB tables
con.execute(f"CREATE TABLE rings AS SELECT * FROM '{ring_file}'")
con.execute(f"CREATE TABLE sidechains AS SELECT * FROM '{sidechain_file}'")
con.execute(f"CREATE TABLE combinations AS SELECT * FROM '{combinations_file}'")

# Query the tables and load results into Pandas DataFrames
df_rings = con.execute("SELECT * FROM rings").fetchdf()
df_sidechains = con.execute("SELECT * FROM sidechains").fetchdf()
df_combinations = con.execute("SELECT * FROM combinations").fetchdf()

# Print the DataFrames
print("Rings DataFrame:")
print(df_rings)

print("\nSidechains DataFrame:")
print(df_sidechains)

print("\nCombinations DataFrame:")
print(df_combinations)

# Close the DuckDB connection
con.close()


IOException: IO Error: No files found that match the pattern "C:\Users\zachg\PycharmProjects\BillionMolecules\billion_mol_db\testing_recombination\data\combinations_2.parquet"

Script to just read the two input files, and generate an empty combinations dataset

In [5]:
import duckdb
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from pathlib import Path

class GenComs_Ring_Sidechains:
    def __init__(self, ring_file, sidechain_file, output_file):
        """
        Initializes the class with input ring and sidechain datasets and the output file path.
        """
        self.ring_file = ring_file
        self.sidechain_file = sidechain_file
        self.output_file = output_file
        self.con = duckdb.connect()

    def load_data(self):
        """
        Loads ring and sidechain datasets into DuckDB tables.
        """
        self.con.execute(f"CREATE TABLE rings AS SELECT * FROM '{self.ring_file}'")
        self.con.execute(f"CREATE TABLE sidechains AS SELECT * FROM '{self.sidechain_file}'")

    def create_empty_parquet(self):
        """
        Creates an empty Parquet file with the correct schema but no data.
        """
        # Define the expected schema based on ring-sidechain combination structure
        parquet_schema = pa.schema([
            ("ring_id", pa.string()),
            ("ring_smiles", pa.string()),
            ("sub1_id", pa.string()),
            ("sub1_smiles", pa.string()),
            ("sub2_id", pa.string()),  # Keeping space for at least 2 sidechains
            ("sub2_smiles", pa.string())
        ])

        # Create an empty Arrow Table
        empty_table = pa.Table.from_pandas(pd.DataFrame(columns=parquet_schema.names))

        # Write the empty table to a Parquet file
        pq.write_table(empty_table, self.output_file)

        print(f"Empty Parquet file created: {self.output_file}")

    def close_connection(self):
        """
        Closes the DuckDB connection.
        """
        self.con.close()

# Example usage
if __name__ == "__main__":
    # Define file paths
    HOME_DIR = Path.cwd()
    ring_file = HOME_DIR / "data" / "ring_list.parquet"
    sidechain_file = HOME_DIR / "data" / "sidechain_list.parquet"
    output_file = HOME_DIR / "data" / "combinations.parquet"

    # Initialize and run the class
    generator = GenComs_Ring_Sidechains(ring_file, sidechain_file, output_file)
    generator.load_data()
    generator.create_empty_parquet()
    generator.close_connection()


Empty Parquet file created: C:\Users\zachg\PycharmProjects\BillionMolecules\billion_mol_db\testing_recombination\data\combinations.parquet


In [9]:
import duckdb
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from pathlib import Path
import itertools
import os

class GenComs_Ring_Sidechains:
    def __init__(self, ring_file, sidechain_file, output_file):
        """
        Initializes the class with input ring and sidechain datasets and the output file path.
        """
        self.ring_file = ring_file
        self.sidechain_file = sidechain_file
        self.output_file = output_file
        self.con = duckdb.connect()

    def load_data(self):
        """
        Loads ring and sidechain datasets into DuckDB tables.
        """
        self.con.execute(f"CREATE TABLE rings AS SELECT * FROM '{self.ring_file}'")
        self.con.execute(f"CREATE TABLE sidechains AS SELECT * FROM '{self.sidechain_file}'")

    def ensure_valid_parquet(self):
        """
        Ensures that the output Parquet file is valid by creating an empty file with the correct schema if needed.
        """
        parquet_schema = pa.schema([
            ("ring_id", pa.string()),
            ("ring_smiles", pa.string()),
            ("sub1_id", pa.string()),
            ("sub1_smiles", pa.string()),
            ("sub2_id", pa.string()),  # Ensure consistent string type
            ("sub2_smiles", pa.string()),  # Ensure consistent string type
            ("combination", pa.string())
        ])

        # If the file does not exist or is empty, create a valid empty Parquet file
        if not os.path.exists(self.output_file) or os.path.getsize(self.output_file) == 0:
            print(f"Initializing {self.output_file} with an empty valid schema.")
            empty_table = pa.Table.from_pandas(pd.DataFrame(columns=parquet_schema.names), schema=parquet_schema)
            pq.write_table(empty_table, self.output_file)

    def generate_combinations(self):
        """
        Processes rings one at a time, generating all valid combinations with sidechains,
        and saves them iteratively to `combinations.parquet`.
        """
        # Ensure output file is valid before writing
        self.ensure_valid_parquet()

        # Read rings table into DuckDB
        rings_df = self.con.execute("SELECT * FROM rings").fetchdf()
        
        # Read sidechains table into DuckDB
        sidechains_df = self.con.execute("SELECT * FROM sidechains").fetchdf()

        # Process one ring at a time to prevent memory overload
        for _, ring in rings_df.iterrows():
            ring_id = ring["id"]
            ring_smiles = ring["smiles"]
            n_subs = ring["n_subs"]

            # Generate all possible combinations of `n_subs` sidechains
            sidechain_combinations = list(itertools.combinations(sidechains_df.itertuples(index=False), n_subs))

            results = []
            for sidechain_set in sidechain_combinations:
                sub_ids = [sc.id for sc in sidechain_set]
                sub_smiles = [sc.smiles for sc in sidechain_set]

                # Create the new combination SMILES string
                combination_smiles = ring_smiles + "(" + ")(".join(sub_smiles) + ")"

                # Construct a row, ensuring all fields are explicitly strings
                row = {
                    "ring_id": str(ring_id),
                    "ring_smiles": str(ring_smiles),
                    "sub1_id": str(sub_ids[0]) if len(sub_ids) > 0 else "",
                    "sub1_smiles": str(sub_smiles[0]) if len(sub_smiles) > 0 else "",
                    "sub2_id": str(sub_ids[1]) if len(sub_ids) > 1 else "",
                    "sub2_smiles": str(sub_smiles[1]) if len(sub_smiles) > 1 else "",
                    "combination": str(combination_smiles)
                }
                results.append(row)

            # Convert results to a DataFrame with explicit types
            df_combinations = pd.DataFrame(results, columns=["ring_id", "ring_smiles", "sub1_id", "sub1_smiles",
                                                             "sub2_id", "sub2_smiles", "combination"])
            df_combinations = df_combinations.astype(str)  # Ensure consistent string types

            # Append to Parquet file manually
            if not df_combinations.empty:
                # Read the existing Parquet file
                existing_table = pq.read_table(self.output_file)

                # Convert DataFrame to Arrow Table with the same schema
                new_table = pa.Table.from_pandas(df_combinations, schema=existing_table.schema)

                # Concatenate existing and new tables
                combined_table = pa.concat_tables([existing_table, new_table])

                # Write back to Parquet file
                pq.write_table(combined_table, self.output_file)

            print(f"Processed ring {ring_id} with {n_subs} substituent positions.")

    def close_connection(self):
        """
        Closes the DuckDB connection.
        """
        self.con.close()

# Example usage
if __name__ == "__main__":
    # Define file paths
    HOME_DIR = Path.cwd()
    ring_file = HOME_DIR / "data" / "ring_list.parquet"
    sidechain_file = HOME_DIR / "data" / "sidechain_list.parquet"
    output_file = HOME_DIR / "data" / "combinations.parquet"

    # Initialize and run the class
    generator = GenComs_Ring_Sidechains(ring_file, sidechain_file, output_file)
    generator.load_data()
    generator.generate_combinations()
    generator.close_connection()


Processed ring R1 with 2 substituent positions.
Processed ring R2 with 3 substituent positions.
Processed ring R3 with 1 substituent positions.


In [64]:
import duckdb
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from pathlib import Path
import itertools
import os
import numpy as np

class GenComs_Ring_Sidechains:
    def __init__(self, ring_file, sidechain_file, output_dir):
        """
        Initializes the class with input ring and sidechain datasets and the output directory.
        The Parquet file name will be automatically determined to avoid overwrites.
        """
        self.ring_file = ring_file
        self.sidechain_file = sidechain_file
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.output_file = self.get_unique_filename()
        self.con = duckdb.connect()

    def get_unique_filename(self):
        """
        Generate a unique Parquet filename by appending _1, _2, etc., if a file already exists.
        """
        base_filename = "combinations"
        extension = ".parquet"
        output_path = self.output_dir / f"{base_filename}{extension}"

        counter = 1
        while output_path.exists():
            output_path = self.output_dir / f"{base_filename}_{counter}{extension}"
            counter += 1

        return output_path

    def load_data(self):
        """
        Loads ring and sidechain datasets into DuckDB tables.
        """
        self.con.execute(f"CREATE TABLE rings AS SELECT * FROM '{self.ring_file}'")
        self.con.execute(f"CREATE TABLE sidechains AS SELECT * FROM '{self.sidechain_file}'")

    def ensure_valid_parquet(self):
        """
        Ensures that the output Parquet file is valid by creating an empty file with the correct schema if needed.
        """
        parquet_schema = pa.schema([
            ("ring_id", pa.string()),
            ("ring_smiles", pa.string()),
            ("sub_id", pa.list_(pa.string())),  # Ordered list of sidechain IDs
            ("sub_smiles", pa.list_(pa.string())),  # Ordered list of sidechain SMILES
            ("combination", pa.string())  # Placeholder for now
        ])

        # If the file does not exist or is empty, create a valid empty Parquet file
        if not os.path.exists(self.output_file) or os.path.getsize(self.output_file) == 0:
            print(f"Initializing {self.output_file} with an empty valid schema.")
            empty_table = pa.Table.from_pandas(pd.DataFrame(columns=parquet_schema.names), schema=parquet_schema)
            pq.write_table(empty_table, self.output_file)

    def generate_combinations(self):
        """
        Processes rings one at a time, generating all possible sidechain combinations
        (including repeated sidechains) based on the number of substitution points.
        Saves results to the Parquet file iteratively.
        """
        # Ensure output file is valid before writing
        self.ensure_valid_parquet()

        # Read rings and sidechains tables into DuckDB
        rings_df = self.con.execute("SELECT * FROM rings").fetchdf()
        sidechains_df = self.con.execute("SELECT * FROM sidechains").fetchdf()

        # Define the output Parquet schema
        parquet_schema = pa.schema([
            ("ring_id", pa.string()),
            ("ring_smiles", pa.string()),
            ("sub_id", pa.list_(pa.string())),  # Ordered list of sidechain IDs
            ("sub_smiles", pa.list_(pa.string())),  # Ordered list of sidechain SMILES
            ("combination", pa.string())  # Placeholder for now
        ])

        # Process one ring at a time
        for _, ring in rings_df.iterrows():
            ring_id = ring["id"]
            ring_smiles = ring["smiles"]
            n_subs = ring["n_subs"]

            # Generate all possible combinations of `n_subs` sidechains (including repetitions)
            sidechain_combinations = list(itertools.product(sidechains_df.itertuples(index=False), repeat=n_subs))

            results = []
            for sidechain_set in sidechain_combinations:
                sub_ids = [sc.id for sc in sidechain_set]
                sub_smiles = [sc.smiles for sc in sidechain_set]

                # Construct a row with ordered lists
                row = {
                    "ring_id": str(ring_id),
                    "ring_smiles": str(ring_smiles),
                    "sub_id": sub_ids,  # List of sidechain IDs
                    "sub_smiles": sub_smiles,  # List of sidechain SMILES
                    "combination": "N/A"  # Placeholder
                }
                results.append(row)

            # Convert results to a DataFrame
            df_combinations = pd.DataFrame(results,
                                           columns=["ring_id", "ring_smiles", "sub_id", "sub_smiles", "combination"])

            # Append to Parquet file manually
            if not df_combinations.empty:
                # Read the existing Parquet file
                existing_table = pq.read_table(self.output_file)

                # Convert DataFrame to Arrow Table with the same schema
                new_table = pa.Table.from_pandas(df_combinations, schema=parquet_schema)

                # Concatenate existing and new tables
                combined_table = pa.concat_tables([existing_table, new_table])

                # Write back to Parquet file
                pq.write_table(combined_table, self.output_file)

            print(f"Processed ring {ring_id} with {n_subs} substituent positions.")

    def close_connection(self):
        """
        Closes the DuckDB connection.
        """
        self.con.close()
    
# Example usage
if __name__ == "__main__":
    # Define file paths
    HOME_DIR = Path.cwd()
    ring_file = HOME_DIR / "data" / "ring_list.parquet"
    sidechain_file = HOME_DIR / "data" / "sidechain_list.parquet"
    output_dir = HOME_DIR / "output"  # Store files in an output directory

    # Initialize and run the class
    generator = GenComs_Ring_Sidechains(ring_file, sidechain_file, output_dir)
    generator.load_data()
    generator.generate_combinations()
    generator.close_connection()


Initializing C:\Users\zachg\PycharmProjects\BillionMolecules\billion_mol_db\testing_recombination\output\combinations.parquet with an empty valid schema.
Processed ring R1 with 2 substituent positions.
Processed ring R2 with 3 substituent positions.
Processed ring R3 with 1 substituent positions.
Processed ring R3 with 5 substituent positions.
Processed ring R3 with 4 substituent positions.
Processed ring R3 with 4 substituent positions.
Processed ring R3 with 4 substituent positions.
Processed ring R3 with 4 substituent positions.
Processed ring R3 with 4 substituent positions.
Processed ring R3 with 4 substituent positions.


In [44]:
import duckdb
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from pathlib import Path
import itertools
import os


class GenComs_Ring_Sidechains:
    def __init__(self, ring_file, sidechain_file, output_dir):
        """
        Initializes the class with input ring and sidechain datasets and the output directory.
        The Parquet file name will be automatically determined to avoid overwrites.
        """
        self.ring_file = ring_file
        self.sidechain_file = sidechain_file
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.output_file = self.get_unique_filename()
        self.con = duckdb.connect()

    def get_unique_filename(self):
        """
        Generate a unique Parquet filename by appending _1, _2, etc., if a file already exists.
        """
        base_filename = "combinations"
        extension = ".parquet"
        output_path = self.output_dir / f"{base_filename}{extension}"

        counter = 1
        while output_path.exists():
            output_path = self.output_dir / f"{base_filename}_{counter}{extension}"
            counter += 1

        return output_path

    def load_data(self):
        """
        Loads ring and sidechain datasets into DuckDB tables.
        """
        self.con.execute(f"CREATE TABLE rings AS SELECT * FROM '{self.ring_file}'")
        self.con.execute(f"CREATE TABLE sidechains AS SELECT * FROM '{self.sidechain_file}'")

    def ensure_valid_parquet(self):
        """
        Ensures that the output Parquet file is valid by creating an empty file with the correct schema if needed.
        """
        parquet_schema = pa.schema([
            ("ring_id", pa.string()),
            ("ring_smiles", pa.string()),
            ("sub_id", pa.list_(pa.string())),  # Ordered list of sidechain IDs
            ("sub_smiles", pa.list_(pa.string())),  # Ordered list of sidechain SMILES
            ("combination", pa.string())  # Placeholder for now
        ])

        # If the file does not exist or is empty, create a valid empty Parquet file
        if not os.path.exists(self.output_file) or os.path.getsize(self.output_file) == 0:
            print(f"Initializing {self.output_file} with an empty valid schema.")
            empty_table = pa.Table.from_pandas(pd.DataFrame(columns=parquet_schema.names), schema=parquet_schema)
            pq.write_table(empty_table, self.output_file)

    def generate_combinations(self):
        """
        Processes rings one at a time, generating all possible sidechain combinations
        (including repeated sidechains) based on the number of substitution points.
        Saves results to the Parquet file iteratively.
        """
        # Ensure output file is valid before writing
        self.ensure_valid_parquet()

        # Read rings and sidechains tables into DuckDB
        rings_df = self.con.execute("SELECT * FROM rings").fetchdf()
        sidechains_df = self.con.execute("SELECT * FROM sidechains").fetchdf()

        # Define the output Parquet schema
        parquet_schema = pa.schema([
            ("ring_id", pa.string()),
            ("ring_smiles", pa.string()),
            ("sub_id", pa.list_(pa.string())),  # Ordered list of sidechain IDs
            ("sub_smiles", pa.list_(pa.string())),  # Ordered list of sidechain SMILES
            ("combination", pa.string())  # Placeholder for now
        ])

        # Process one ring at a time
        for _, ring in rings_df.iterrows():
            ring_id = ring["id"]
            ring_smiles = ring["subgraph_smiles"]
            n_subs = ring["sub_points"]

            # Generate all possible combinations of `n_subs` sidechains (including repetitions)
            sidechain_combinations = list(itertools.product(sidechains_df.itertuples(index=False), repeat=n_subs))

            results = []
            for sidechain_set in sidechain_combinations:
                sub_ids = [str(sc.id) for sc in sidechain_set]  # Convert IDs to strings
                sub_smiles = [sc.smiles for sc in sidechain_set]

                # Construct a row with ordered lists
                row = {
                    "ring_id": str(ring_id),
                    "ring_smiles": str(ring_smiles),
                    "sub_id": sub_ids,  # List of sidechain IDs
                    "sub_smiles": sub_smiles,  # List of sidechain SMILES
                    "combination": "N/A"  # Placeholder
                }
                results.append(row)

            # Convert results to a DataFrame
            df_combinations = pd.DataFrame(results,
                                           columns=["ring_id", "ring_smiles", "sub_id", "sub_smiles", "combination"])

            # Append to Parquet file manually
            if not df_combinations.empty:
                # Read the existing Parquet file
                existing_table = pq.read_table(self.output_file)

                # Convert DataFrame to Arrow Table with the same schema
                new_table = pa.Table.from_pandas(df_combinations, schema=parquet_schema)

                # Concatenate existing and new tables
                combined_table = pa.concat_tables([existing_table, new_table])

                # Write back to Parquet file
                pq.write_table(combined_table, self.output_file)

            print(f"Processed ring {ring_id} with {n_subs} substituent positions.")

    def close_connection(self):
        """
        Closes the DuckDB connection.
        """
        self.con.close()


# Example usage
if __name__ == "__main__":
    # Define file paths
    HOME_DIR = Path.cwd()
    ring_file = HOME_DIR / "data" / "rings.parquet"
    sidechain_file = HOME_DIR / "data" / "substituents.parquet"
    output_dir = HOME_DIR / "output"  # Store files in an output directory

    # Initialize and run the class
    generator = GenComs_Ring_Sidechains(ring_file, sidechain_file, output_dir)
    generator.load_data()
    generator.generate_combinations()
    generator.close_connection()

Initializing C:\Users\zachg\PycharmProjects\BillionMolecules\billion_mol_db\testing_recombination\output\combinations_3.parquet with an empty valid schema.
Processed ring 050b9af8ce90e9f70872fbd81684098d2b7ae6586242b7983a8d41831f42a40d with 4 substituent positions.
Processed ring 107d1460153a3251a50f08e42f8c91163a80de99e81fa5702eb37d130b706db8 with 4 substituent positions.
Processed ring f55dd7aa7ecb253991b8728db9a2cbfbde48b2ebb0dd8193c344e14bd36ef95e with 2 substituent positions.
Processed ring 65d157cac81254b7dfcad12d72817b81183017e347a682362eeec98f4a6a9a8a with 3 substituent positions.
Processed ring c07be7d8ce64fa3153709a72cfdb3fd9b6ce663b26a62e5700cbe801b0a5540e with 2 substituent positions.
Processed ring 31bd7eb0d5abbb236239b38045782c351a5b6ccba144f2f6a1f8d2bae395f18f with 2 substituent positions.
Processed ring 414a50c9d0280dd0d4eef191fe27f8a53be874c4713001c461882caedf95ca99 with 1 substituent positions.


In [111]:
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path

# Define the path to the Parquet file
parquet_file = Path(r"C:\Users\zachg\PycharmProjects\BillionMolecules\billion_mol_db\testing_recombination\output\combinations_trun.parquet")

# Check if the file exists
if not parquet_file.exists():
    print(f"Error: File {parquet_file} does not exist.")
else:
    # Read the Parquet file into a Pandas DataFrame
    df = pd.read_parquet(parquet_file, engine="pyarrow")

    # Print the DataFrame
    print("Loaded DataFrame from Parquet:")
    print(df)


Loaded DataFrame from Parquet:
                                             ring_id              ring_smiles  \
0  050b9af8ce90e9f70872fbd81684098d2b7ae6586242b7...  c1nncc2c1CC1=C(C2)OCCC1   
1  050b9af8ce90e9f70872fbd81684098d2b7ae6586242b7...  c1nncc2c1CC1=C(C2)OCCC1   

                             sub_id                                sub_smiles  \
0  [107999, 107999, 107999, 107999]  [CCC[SeH], CCC[SeH], CCC[SeH], CCC[SeH]]   
1     [107999, 107999, 107999, 129]        [CCC[SeH], CCC[SeH], CCC[SeH], Br]   

                                         combination  row_idx  
0  [SeH]CCCC1C2=C(OC(CCC[SeH])C(CCC[SeH])C2)C(CCC...        1  
1  [SeH]CCCC1C2=C(OC(CCC[SeH])C(CCC[SeH])C2)C(CCC...        2  


In [93]:
import duckdb
import ast
import networkx as nx
from rdkit import Chem

# File paths
combinations_file = r"C:\Users\zachg\PycharmProjects\BillionMolecules\billion_mol_db\testing_recombination\output\combinations_trun.parquet"
rings_file = r"C:\Users\zachg\PycharmProjects\BillionMolecules\billion_mol_db\testing_recombination\data\rings.parquet"
substituents_file = r"C:\Users\zachg\PycharmProjects\BillionMolecules\billion_mol_db\testing_recombination\data\substituents.parquet"
Class
def parse_nodes(nodes):
    """Convert node string to list of tuples (index, label)."""
    if isinstance(nodes, str):
        try:
            nodes = ast.literal_eval(nodes)
        except (SyntaxError, ValueError):
            return []
    return [(int(idx), str(label)) for idx, label in nodes if isinstance(idx, int) and isinstance(label, str)]

def parse_edges(edges):
    """Convert edge data to list of tuples (node1, node2, bond type), handling different formats."""
    edge_list = []
    
    if isinstance(edges, str):
        edge_entries = edges.split(", ")
    elif isinstance(edges, list):  # Handle case where edges are stored as a list of strings
        edge_entries = [e.strip("'") for e in edges]  # Remove surrounding single quotes
    else:
        return edge_list  # Return empty if edges are not valid
    
    for edge in edge_entries:
        try:
            nodes_part, bond_type = edge.split(": ")
            node1, node2 = map(int, nodes_part.split(","))
            edge_list.append((node1, node2, bond_type))
        except ValueError:
            continue
    
    return edge_list

def renumber_nodes(ring_nodes, ring_edges, sidechains):
    """Renumber nodes sequentially for the ring and sidechains.
    
    - The ring is renumbered starting at 0.
    - The first sidechain starts immediately after the ring nodes.
    - Each subsequent sidechain continues numbering sequentially.
    """
    node_map = {}
    new_nodes = []
    new_edges = []
    
    # Renumber ring nodes
    current_index = 0
    for old_idx, label in ring_nodes:
        node_map[old_idx] = current_index
        new_nodes.append((current_index, label))
        current_index += 1

    for node1, node2, bond in ring_edges:
        new_edges.append((node_map[node1], node_map[node2], bond))

    renumbered_sidechains = []
    
    # Renumber sidechain nodes
    for sub_nodes, sub_edges in sidechains:
        sub_node_map = {}
        new_sub_nodes = []
        new_sub_edges = []

        for old_idx, label in sub_nodes:
            sub_node_map[old_idx] = current_index
            new_sub_nodes.append((current_index, label))
            current_index += 1

        for node1, node2, bond in sub_edges:
            new_sub_edges.append((sub_node_map[node1], sub_node_map[node2], bond))

        renumbered_sidechains.append((new_sub_nodes, new_sub_edges))

    return new_nodes, new_edges, renumbered_sidechains

def extract_G():
    """Extract the first row from combinations and print the ring and sidechains' nodes and edges."""
    con = duckdb.connect()

    # Get the first row from combinations
    combination_data = con.execute(f"SELECT * FROM read_parquet('{combinations_file}') LIMIT 1").fetchone()
    
    if not combination_data:
        print("No data found in combinations.")
        return None, None, None  # Ensure the function returns None if no data found

    ring_id = combination_data[0]  # Assuming 'ring_id' is the first column
    sub_ids = combination_data[2]  # Assuming 'sub_id' is the third column (list of sidechains)

    # Retrieve the ring graph data
    ring_data = con.execute(f"SELECT nodes, edges FROM read_parquet('{rings_file}') WHERE id = ?", [ring_id]).fetchone()
    
    if not ring_data:
        print(f"Ring {ring_id} not found.\n")
        return None, None, None

    ring_nodes = parse_nodes(ring_data[0])
    ring_edges = parse_edges(ast.literal_eval(ring_data[1]) if isinstance(ring_data[1], str) and ring_data[1].startswith("[") else ring_data[1])

    # Retrieve and store sidechain data
    sidechains = []
    for sub_id in sub_ids:
        sub_data = con.execute(f"SELECT nodes, edges FROM read_parquet('{substituents_file}') WHERE id = ?", [sub_id]).fetchone()
        
        if not sub_data:
            print(f"Substituent {sub_id} not found.\n")
            continue

        sub_nodes = parse_nodes(sub_data[0])
        sub_edges = parse_edges(sub_data[1] if sub_data[1] is not None else "")
        sidechains.append((sub_nodes, sub_edges))

    # Renumber nodes
    new_ring_nodes, new_ring_edges, renumbered_sidechains = renumber_nodes(ring_nodes, ring_edges, sidechains)

    con.close()
    
    # Return renumbered graph data
    return new_ring_nodes, new_ring_edges, renumbered_sidechains

def bind_sc_rings(ring_nodes, ring_edges, sidechains):
    """Builds a full graph with ring and sidechains intact, then iteratively connects the first '*' node in the ring 
    to the first '*' node in each sidechain, ensuring labels and bonds are preserved. Returns the final NetworkX graph."""
    
    # Create NetworkX graph
    G = nx.Graph()

    # Track node mappings to ensure all indices remain intact
    node_map = {}
    current_index = 0

    # Add ring nodes to the graph, storing '*' node positions
    ring_star_nodes = []
    
    for old_idx, label in ring_nodes:
        node_map[old_idx] = current_index
        G.add_node(current_index, label=label)  # Keep label as is

        if '*' in label:
            ring_star_nodes.append(current_index)  # Store '*' nodes for connections

        current_index += 1

    # Add ring edges
    for n1, n2, bond in ring_edges:
        G.add_edge(node_map[n1], node_map[n2], bond=bond)  # Preserve bond types

    # Process sidechains, tracking '*' separately
    sidechain_star_nodes = []
    sidechain_mappings = []

    for sub_nodes, sub_edges in sidechains:
        sub_node_map = {}
        sub_star_nodes = []  # Track '*' nodes within the sidechain

        for old_idx, label in sub_nodes:
            sub_node_map[old_idx] = current_index
            G.add_node(current_index, label=label)

            if '*' in label:
                sub_star_nodes.append(current_index)

            current_index += 1

        # Store the mapping and star nodes for later connections
        sidechain_mappings.append((sub_node_map, sub_star_nodes))

        # Add sidechain edges with bond types
        for n1, n2, bond in sub_edges:
            G.add_edge(sub_node_map[n1], sub_node_map[n2], bond=bond)  # Preserve bond type

    # Iteratively connect '*' nodes
    for sub_node_map, sub_star_nodes in sidechain_mappings:
        if not ring_star_nodes or not sub_star_nodes:
            continue  # Skip if no available '*' nodes

        ring_star = ring_star_nodes.pop(0)  # Get first '*' node in the ring
        sidechain_star = sub_star_nodes.pop(0)  # Get first '*' node in sidechain

        # Create the new bond
        G.add_edge(ring_star, sidechain_star, bond='Single')

        # Remove only ONE `*` from the labels
        G.nodes[ring_star]['label'] = G.nodes[ring_star]['label'].replace('*', '', 1)
        G.nodes[sidechain_star]['label'] = G.nodes[sidechain_star]['label'].replace('*', '', 1)

    # Return the final NetworkX graph
    return G

def G_to_smi(G):
    """Converts a NetworkX molecular graph into an RDKit SMILES string."""

    # Create an editable RDKit molecule
    mol = Chem.RWMol()

    # Map NetworkX nodes to RDKit atom indices
    atom_map = {}

    # Add atoms to RDKit molecule
    for idx, label in G.nodes(data="label"):
        atom_symbol = label.replace('*', '')  # Ensure '*' is removed before adding atoms
        atom = Chem.Atom(atom_symbol)
        atom_idx = mol.AddAtom(atom)
        atom_map[idx] = atom_idx  # Track mapping

    # Add bonds to RDKit molecule
    bond_dict = {'Single': Chem.BondType.SINGLE, 'Double': Chem.BondType.DOUBLE, 'Aromatic': Chem.BondType.AROMATIC}
    
    for n1, n2, bond in G.edges(data="bond"):
        mol.AddBond(atom_map[n1], atom_map[n2], bond_dict.get(bond, Chem.BondType.SINGLE))

    # Convert to SMILES
    smiles = Chem.MolToSmiles(mol, canonical=True)
    return smiles


# Run the function to extract and print the data
new_ring_nodes, new_ring_edges, renumbered_sidechains = extract_G()

# Ensure the returned values are valid before proceeding
if new_ring_nodes is not None and new_ring_edges is not None and renumbered_sidechains is not None:
    G = bind_sc_rings(new_ring_nodes, new_ring_edges, renumbered_sidechains)
else:
    print("Graph data could not be extracted. Exiting.")

# Convert the final graph to SMILES
final_smiles = G_to_smi(G)

# Print the generated SMILES string
print("Final SMILES:", final_smiles)



First Row of combinations: -----

Ring ID: 050b9af8ce90e9f70872fbd81684098d2b7ae6586242b7983a8d41831f42a40d
Renumbered Ring
Nodes:
(0, 'C**')
(1, 'C')
(2, 'C')
(3, 'C')
(4, 'C*')
(5, 'C**')
(6, 'O')
(7, 'C*')
(8, 'C')
(9, 'C')
(10, 'N')
(11, 'N')
(12, 'C')
(13, 'C')

Edges:
(0, 1, 'Single')
(0, 13, 'Single')
(1, 2, 'Double')
(1, 6, 'Single')
(2, 3, 'Single')
(2, 7, 'Single')
(3, 4, 'Single')
(4, 5, 'Single')
(5, 6, 'Single')
(7, 8, 'Single')
(8, 9, 'Aromatic')
(8, 13, 'Aromatic')
(9, 10, 'Aromatic')
(10, 11, 'Aromatic')
(11, 12, 'Aromatic')
(12, 13, 'Aromatic')


Renumbered Sidechain 1:
Nodes:
(14, 'Se')
(15, 'C')
(16, 'C')
(17, 'C*')

Edges:
(14, 15, 'Single')
(15, 16, 'Single')
(16, 17, 'Single')


Renumbered Sidechain 2:
Nodes:
(18, 'Se')
(19, 'C')
(20, 'C')
(21, 'C*')

Edges:
(18, 19, 'Single')
(19, 20, 'Single')
(20, 21, 'Single')


Renumbered Sidechain 3:
Nodes:
(22, 'Se')
(23, 'C')
(24, 'C')
(25, 'C*')

Edges:
(22, 23, 'Single')
(23, 24, 'Single')
(24, 25, 'Single')


Renumbered

In [113]:
import duckdb
import ast
import networkx as nx
from rdkit import Chem

class MoleculeReconstructor:
    def __init__(self, combinations_file, rings_file, substituents_file, index=0):
        """
        Initializes the MoleculeReconstructor with file paths and a row index.

        :param combinations_file: Path to the Parquet file containing molecular combinations.
        :param rings_file: Path to the Parquet file containing ring structures.
        :param substituents_file: Path to the Parquet file containing substituents.
        :param index: The row index in combinations_file to process.
        """
        self.combinations_file = combinations_file
        self.rings_file = rings_file
        self.substituents_file = substituents_file
        self.index = index

    def parse_nodes(nodes):
        """Convert node string to list of tuples (index, label)."""
        if isinstance(nodes, str):
            try:
                nodes = ast.literal_eval(nodes)
            except (SyntaxError, ValueError):
                return []
        return [(int(idx), str(label)) for idx, label in nodes if isinstance(idx, int) and isinstance(label, str)]
    
    def parse_edges(edges):
        """Convert edge data to list of tuples (node1, node2, bond type), handling different formats."""
        edge_list = []
        
        if isinstance(edges, str):
            edge_entries = edges.split(", ")
        elif isinstance(edges, list):  # Handle case where edges are stored as a list of strings
            edge_entries = [e.strip("'") for e in edges]  # Remove surrounding single quotes
        else:
            return edge_list  # Return empty if edges are not valid
        
        for edge in edge_entries:
            try:
                nodes_part, bond_type = edge.split(": ")
                node1, node2 = map(int, nodes_part.split(","))
                edge_list.append((node1, node2, bond_type))
            except ValueError:
                continue
        
        return edge_list
    
    def renumber_nodes(ring_nodes, ring_edges, sidechains):
        """Renumber nodes sequentially for the ring and sidechains.
        
        - The ring is renumbered starting at 0.
        - The first sidechain starts immediately after the ring nodes.
        - Each subsequent sidechain continues numbering sequentially.
        """
        node_map = {}
        new_nodes = []
        new_edges = []
        
        # Renumber ring nodes
        current_index = 0
        for old_idx, label in ring_nodes:
            node_map[old_idx] = current_index
            new_nodes.append((current_index, label))
            current_index += 1
    
        for node1, node2, bond in ring_edges:
            new_edges.append((node_map[node1], node_map[node2], bond))
    
        renumbered_sidechains = []
        
        # Renumber sidechain nodes
        for sub_nodes, sub_edges in sidechains:
            sub_node_map = {}
            new_sub_nodes = []
            new_sub_edges = []
    
            for old_idx, label in sub_nodes:
                sub_node_map[old_idx] = current_index
                new_sub_nodes.append((current_index, label))
                current_index += 1
    
            for node1, node2, bond in sub_edges:
                new_sub_edges.append((sub_node_map[node1], sub_node_map[node2], bond))
    
            renumbered_sidechains.append((new_sub_nodes, new_sub_edges))
    
        return new_nodes, new_edges, renumbered_sidechains
    
    def extract_G(self):
        """Extract the first row from combinations and print the ring and sidechains' nodes and edges."""
        con = duckdb.connect()
    
        # Get the first row from combinations
        combination_data = con.execute(f"SELECT * FROM read_parquet('{combinations_file}') LIMIT 1").fetchone()
        
        if not combination_data:
            print("No data found in combinations.")
            return None, None, None  # Ensure the function returns None if no data found
    
        ring_id = combination_data[0]  # Assuming 'ring_id' is the first column
        sub_ids = combination_data[2]  # Assuming 'sub_id' is the third column (list of sidechains)
    
        # Retrieve the ring graph data
        ring_data = con.execute(f"SELECT nodes, edges FROM read_parquet('{rings_file}') WHERE id = ?", [ring_id]).fetchone()
        
        if not ring_data:
            print(f"Ring {ring_id} not found.\n")
            return None, None, None
    
        ring_nodes = parse_nodes(ring_data[0])
        ring_edges = parse_edges(ast.literal_eval(ring_data[1]) if isinstance(ring_data[1], str) and ring_data[1].startswith("[") else ring_data[1])
    
        # Retrieve and store sidechain data
        sidechains = []
        for sub_id in sub_ids:
            sub_data = con.execute(f"SELECT nodes, edges FROM read_parquet('{substituents_file}') WHERE id = ?", [sub_id]).fetchone()
            
            if not sub_data:
                print(f"Substituent {sub_id} not found.\n")
                continue
    
            sub_nodes = parse_nodes(sub_data[0])
            sub_edges = parse_edges(sub_data[1] if sub_data[1] is not None else "")
            sidechains.append((sub_nodes, sub_edges))
    
        # Renumber nodes
        new_ring_nodes, new_ring_edges, renumbered_sidechains = renumber_nodes(ring_nodes, ring_edges, sidechains)
    
        con.close()
        
        # Return renumbered graph data
        return new_ring_nodes, new_ring_edges, renumbered_sidechains
    
    @staticmethod
    def bind_sc_rings(ring_nodes, ring_edges, sidechains):
        """Builds a full graph with ring and sidechains intact, then iteratively connects the first '*' node in the ring 
        to the first '*' node in each sidechain, ensuring labels and bonds are preserved. Returns the final NetworkX graph."""
        
        # Create NetworkX graph
        G = nx.Graph()
    
        # Track node mappings to ensure all indices remain intact
        node_map = {}
        current_index = 0
    
        # Add ring nodes to the graph, storing '*' node positions
        ring_star_nodes = []
        
        for old_idx, label in ring_nodes:
            node_map[old_idx] = current_index
            G.add_node(current_index, label=label)  # Keep label as is
    
            if '*' in label:
                ring_star_nodes.append(current_index)  # Store '*' nodes for connections
    
            current_index += 1
    
        # Add ring edges
        for n1, n2, bond in ring_edges:
            G.add_edge(node_map[n1], node_map[n2], bond=bond)  # Preserve bond types
    
        # Process sidechains, tracking '*' separately
        sidechain_star_nodes = []
        sidechain_mappings = []
    
        for sub_nodes, sub_edges in sidechains:
            sub_node_map = {}
            sub_star_nodes = []  # Track '*' nodes within the sidechain
    
            for old_idx, label in sub_nodes:
                sub_node_map[old_idx] = current_index
                G.add_node(current_index, label=label)
    
                if '*' in label:
                    sub_star_nodes.append(current_index)
    
                current_index += 1
    
            # Store the mapping and star nodes for later connections
            sidechain_mappings.append((sub_node_map, sub_star_nodes))
    
            # Add sidechain edges with bond types
            for n1, n2, bond in sub_edges:
                G.add_edge(sub_node_map[n1], sub_node_map[n2], bond=bond)  # Preserve bond type
    
        # Iteratively connect '*' nodes
        for sub_node_map, sub_star_nodes in sidechain_mappings:
            if not ring_star_nodes or not sub_star_nodes:
                continue  # Skip if no available '*' nodes
    
            ring_star = ring_star_nodes.pop(0)  # Get first '*' node in the ring
            sidechain_star = sub_star_nodes.pop(0)  # Get first '*' node in sidechain
    
            # Create the new bond
            G.add_edge(ring_star, sidechain_star, bond='Single')
    
            # Remove only ONE `*` from the labels
            G.nodes[ring_star]['label'] = G.nodes[ring_star]['label'].replace('*', '', 1)
            G.nodes[sidechain_star]['label'] = G.nodes[sidechain_star]['label'].replace('*', '', 1)
    
        # Return the final NetworkX graph
        return G
    
    @staticmethod
    def G_to_smi(G):
        """Converts a NetworkX molecular graph into an RDKit SMILES string."""
    
        # Create an editable RDKit molecule
        mol = Chem.RWMol()
    
        # Map NetworkX nodes to RDKit atom indices
        atom_map = {}
    
        # Add atoms to RDKit molecule
        for idx, label in G.nodes(data="label"):
            atom_symbol = label.replace('*', '')  # Ensure '*' is removed before adding atoms
            atom = Chem.Atom(atom_symbol)
            atom_idx = mol.AddAtom(atom)
            atom_map[idx] = atom_idx  # Track mapping
    
        # Add bonds to RDKit molecule
        bond_dict = {'Single': Chem.BondType.SINGLE, 'Double': Chem.BondType.DOUBLE, 'Aromatic': Chem.BondType.AROMATIC}
        
        for n1, n2, bond in G.edges(data="bond"):
            mol.AddBond(atom_map[n1], atom_map[n2], bond_dict.get(bond, Chem.BondType.SINGLE))
    
        # Convert to SMILES
        smiles = Chem.MolToSmiles(mol, canonical=True)
        return smiles

    def reconstruct(self):
        """Runs the full reconstruction process, returns a SMILES string, and updates the combinations file using DuckDB."""
        new_ring_nodes, new_ring_edges, renumbered_sidechains = self.extract_G()
    
        if new_ring_nodes is None:
            return "Failed to extract molecular graph."
    
        G = self.bind_sc_rings(new_ring_nodes, new_ring_edges, renumbered_sidechains)
        final_smiles = self.G_to_smi(G)
    
        with duckdb.connect() as con:
            # Step 1: Create a temporary table with a row index
            con.execute(f"""
                CREATE TABLE temp_combinations AS
                SELECT *, ROW_NUMBER() OVER () AS row_idx
                FROM read_parquet('{self.combinations_file}');
            """)
    
            # Step 2: Update the row where row_idx matches self.index + 1 (DuckDB uses 1-based indexing)
            con.execute(
                f"""
                UPDATE temp_combinations
                SET combination = ?
                WHERE row_idx = ?;
                """,
                [final_smiles, self.index + 1]  # Convert Python's 0-based index to DuckDB's 1-based index
            )
    
            # Step 3: Write the updated table back to the Parquet file
            con.execute(f"COPY temp_combinations TO '{self.combinations_file}' (FORMAT 'parquet');")
    
        print(f"Updated row {self.index} in combinations file with SMILES: {final_smiles}")
    
        return final_smiles






# Usage Example
if __name__ == "__main__":
    combinations_file = r"C:\Users\zachg\PycharmProjects\BillionMolecules\billion_mol_db\testing_recombination\output\combinations_trun.parquet"
    rings_file = r"C:\Users\zachg\PycharmProjects\BillionMolecules\billion_mol_db\testing_recombination\data\rings.parquet"
    substituents_file = r"C:\Users\zachg\PycharmProjects\BillionMolecules\billion_mol_db\testing_recombination\data\substituents.parquet"

    reconstructor = MoleculeReconstructor(combinations_file, rings_file, substituents_file, index=2)
    final_smiles = reconstructor.reconstruct()
    print("Final SMILES:", final_smiles)


Updated row 2 in combinations file with SMILES: [SeH]CCCC1C2=C(OC(CCC[SeH])C(CCC[SeH])C2)C(CCC[SeH])c2cnncc21
Final SMILES: [SeH]CCCC1C2=C(OC(CCC[SeH])C(CCC[SeH])C2)C(CCC[SeH])c2cnncc21


In [50]:
import pandas as pd
from pathlib import Path

# Define file paths
input_parquet = Path(r"C:\Users\zachg\PycharmProjects\BillionMolecules\billion_mol_db\testing_recombination\output\combinations.parquet")
output_parquet = Path(r"C:\Users\zachg\PycharmProjects\BillionMolecules\billion_mol_db\testing_recombination\output\combinations_trun.parquet")

# Load the Parquet file
df = pd.read_parquet(input_parquet)

# Keep only the first two rows
df_truncated = df.iloc[:2]

# Save as a new Parquet file
df_truncated.to_parquet(output_parquet, engine="pyarrow")

print(f"Saved truncated combinations file at {output_parquet}")


Saved truncated combinations file at C:\Users\zachg\PycharmProjects\BillionMolecules\billion_mol_db\testing_recombination\output\combinations_trun.parquet


In [115]:
import pandas as pd

# Define file paths
substituents_file = r"C:\Users\zachg\PycharmProjects\BillionMolecules\billion_mol_db\testing_recombination\data\substituents.parquet"
rings_file = r"C:\Users\zachg\PycharmProjects\BillionMolecules\billion_mol_db\testing_recombination\data\rings.parquet"

# Load Parquet files into Pandas DataFrames
df_substituents = pd.read_parquet(substituents_file)
df_rings = pd.read_parquet(rings_file)

# Display the first few rows of each DataFrame
print("Combinations DataFrame:")
print(df_substituents.head())

print("\nRings DataFrame:")
print(df_rings.head())


Combinations DataFrame:
       id  parent_id    smiles                                       nodes  \
0  107999      17024  CCC[SeH]  [(1, 'Se'), (2, 'C'), (3, 'C'), (0, 'C*')]   
1     129         68        Br                                [(0, 'Br*')]   
2   86023      14007    C#CC=N   [(1, 'N'), (2, 'C'), (3, 'C'), (0, 'C*')]   
3    5472       1533    [SiH4]                                [(0, 'Si*')]   
4    8812       2217     C=CCl            [(1, 'C'), (0, 'C*'), (2, 'Cl')]   

                                   edges  heavy_atoms  atom_C  atom_O  atom_S  \
0  1,2: Single, 2,3: Single, 3,0: Single            4       3       0       0   
1                                   None            1       0       0       0   
2  1,2: Double, 2,3: Single, 3,0: Triple            4       3       0       0   
3                                   None            1       0       0       0   
4               1,0: Double, 0,2: Single            3       2       0       0   

   atom_N  ...  atom

In [116]:
import pandas as pd

# Define file path
rings_file = r"C:\Users\zachg\PycharmProjects\BillionMolecules\billion_mol_db\testing_recombination\data\rings.parquet"

# Load the Parquet file into a DataFrame
df_rings = pd.read_parquet(rings_file)

# Ensure nodes are converted to string and count '*' occurrences
df_rings["nodes_str"] = df_rings["nodes"].astype(str)  # Convert nodes to string
df_rings["star_count"] = df_rings["nodes_str"].apply(lambda x: x.count('*'))  # Count '*' occurrences

# Display the first few rows with the new column
print(df_rings[["nodes_str", "star_count"]].head())

# Optional: Get summary statistics
print("\nSummary Statistics for '*' Count:")
print(df_rings["star_count"].describe())


                                              nodes_str  star_count
5159  [(12, 'C**'), (14, 'C'), (15, 'C'), (16, 'C'),...           6
3206  [(3, 'C*'), (4, 'O'), (5, 'C*'), (7, 'C'), (8,...           4
3626  [(2, 'C*'), (3, 'N'), (4, 'S*'), (6, 'C'), (7,...           2
3708  [(2, 'C*'), (3, 'C'), (4, 'C'), (5, 'C'), (6, ...           3
5171  [(4, 'C*'), (5, 'C'), (6, 'C'), (7, 'C'), (8, ...           2

Summary Statistics for '*' Count:
count    7.000000
mean     3.000000
std      1.632993
min      1.000000
25%      2.000000
50%      3.000000
75%      3.500000
max      6.000000
Name: star_count, dtype: float64
